# 1:1 Replication: (Re-)Imag(in)ing Price Trends (M4 Mac 16GB Optimized)
This notebook is a full replication of the experiments from "(Re-)Imag(in)ing Price Trends" (Jiang, Kelly, Xiu 2023).
**Mac M4 16GB Optimization:** This notebook has been specifically refactored to use "Lazy Loading." Instead of generating 7.5 million images up-front (which would require ~115 GB of RAM and crash your Mac), it stores only lightweight data indices and generates images on-the-fly using the M4's blazing fast CPU cores, feeding them directly to the `mps` GPU engine.


In [1]:
import yfinance as yf
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image, ImageDraw
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from tqdm.auto import tqdm
import time
import os
import requests
import warnings
warnings.filterwarnings('ignore')

# The M4 chip relies on Metal Performance Shaders (MPS) for GPU acceleration.
device = torch.device('cuda' if torch.cuda.is_available() else 'mps' if torch.backends.mps.is_available() else 'cpu')
print(f"Using device: {device}")


Using device: mps


## 1. Data Loading
Loading the preprocessed S&P 500 dataset containing OHLCV and Baseline Signals (MOM, STR, WSTR, TREND).


In [2]:
import os

# SET TESTING=False for full replication (Warning: Takes days to train)
TESTING = False 

file_name = 'market_data_sp500_1997_2023.parquet'
if not os.path.exists(file_name):
    raise FileNotFoundError(f"{file_name} not found! Please run the Data Prep notebook first.")

df_market = pd.read_parquet(file_name)
print(f"Loaded {len(df_market)} rows from {file_name}")

if TESTING:
    print("TESTING MODE ON: Subsetting to just 3 tickers for a quick dry-run.")
    test_tickers = df_market['Ticker'].unique()[:3]
    df_market = df_market[df_market['Ticker'].isin(test_tickers)].copy()


Loaded 2739387 rows from market_data_sp500_1997_2023.parquet


## 3. Dataset and Dataloader
We use a standalone Python module `mac_dataset.py` to define the LazyStockImageDataset. This allows us to use PyTorch multiprocessing (`num_workers > 0`) on macOS, which requires external modules to spawn parallel workers.


In [3]:
from mac_dataset import LazyStockImageDataset

def create_dataloaders(df, n_days, horizon, batch_size=128):
    train_val_df = df[(df['Date'] >= '1997-01-01') & (df['Date'] <= '2004-12-31')]
    test_df = df[(df['Date'] >= '2005-01-01') & (df['Date'] <= '2023-12-31')]
    
    train_val_ds = LazyStockImageDataset(train_val_df, n_days, horizon)
    
    if len(train_val_ds) == 0: return None, None, None
        
    train_size = int(0.7 * len(train_val_ds))
    val_size = len(train_val_ds) - train_size
    train_ds, val_ds = torch.utils.data.random_split(train_val_ds, [train_size, val_size])
    
    test_ds = LazyStockImageDataset(test_df, n_days, horizon)
    
    # Using num_workers=8 to fully saturate the M4 CPU cores, keeping the MPS GPU fed.
    # pin_memory=True and prefetch_factor=2 help stream data to the GPU asynchronously.
    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True, num_workers=8, pin_memory=True, prefetch_factor=2)
    val_loader = DataLoader(val_ds, batch_size=batch_size, shuffle=False, num_workers=8, pin_memory=True, prefetch_factor=2)
    test_loader = DataLoader(test_ds, batch_size=batch_size, shuffle=False, num_workers=8, pin_memory=True, prefetch_factor=2)
    
    return train_loader, val_loader, test_loader


## 5. CNN Architecture (PyTorch)


In [4]:
def xavier_init(m):
    if isinstance(m, nn.Conv2d) or isinstance(m, nn.Linear):
        nn.init.xavier_uniform_(m.weight)
        if m.bias is not None:
            nn.init.zeros_(m.bias)

class TrendCNN(nn.Module):
    def __init__(self, n_days=20):
        super(TrendCNN, self).__init__()
        
        if n_days == 5: blocks, v_stride, v_dilation = 2, 1, 1
        elif n_days == 20: blocks, v_stride, v_dilation = 3, 3, 2
        else: blocks, v_stride, v_dilation = 4, 3, 3
            
        layers = []
        in_channels, out_channels = 1, 64
        
        for i in range(blocks):
            stride = (v_stride, 1) if i == 0 else (1, 1)
            dilation = (v_dilation, 1) if i == 0 else (1, 1)
            layers.append(nn.Conv2d(in_channels, out_channels, kernel_size=(5, 3), stride=stride, padding=(2, 1), dilation=dilation))
            layers.append(nn.BatchNorm2d(out_channels))
            layers.append(nn.LeakyReLU(0.01))
            layers.append(nn.MaxPool2d(kernel_size=(2, 1)))
            in_channels = out_channels
            out_channels *= 2
            
        self.features = nn.Sequential(*layers)
        
        test_tensor = torch.zeros(1, 1, 32 if n_days==5 else (64 if n_days==20 else 96), n_days * 3)
        flat_size = self.features(test_tensor).view(1, -1).size(1)
        
        self.classifier = nn.Sequential(
            nn.Dropout(0.5),
            nn.Linear(flat_size, 2)
        )
        self.apply(xavier_init)
        
    def forward(self, x):
        x = self.features(x)
        x = x.view(x.size(0), -1)
        x = self.classifier(x)
        return x


## 6. Training & Evaluation Engine


In [5]:
def train_model(model, train_loader, val_loader, epochs=100, lr=1e-5, patience=2):
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=lr)
    model.to(device)
    
    best_loss = float('inf')
    patience_counter = 0
    
    for epoch in range(epochs):
        model.train()
        train_loss = 0
        for imgs, labels, _ in train_loader:
            imgs, labels = imgs.to(device, non_blocking=True), labels.to(device, non_blocking=True)
            optimizer.zero_grad(set_to_none=True)
            outputs = model(imgs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            train_loss += loss.item()
            
        model.eval()
        val_loss = 0
        with torch.no_grad():
            for imgs, labels, _ in val_loader:
                imgs, labels = imgs.to(device, non_blocking=True), labels.to(device, non_blocking=True)
                outputs = model(imgs)
                loss = criterion(outputs, labels)
                val_loss += loss.item()
                
        train_loss /= len(train_loader)
        val_loss /= len(val_loader)
        
        if val_loss < best_loss:
            best_loss = val_loss
            patience_counter = 0
            torch.save(model.state_dict(), 'best_model.pt')
        else:
            patience_counter += 1
            if patience_counter >= patience: break
                
    try: model.load_state_dict(torch.load('best_model.pt', weights_only=True))
    except: model.load_state_dict(torch.load('best_model.pt'))
    return model

def generate_predictions(model, test_loader):
    model.eval()
    model.to(device)
    predictions, baselines_list = [], []
    with torch.no_grad():
        for imgs, _, baselines in test_loader:
            imgs = imgs.to(device, non_blocking=True)
            probs = torch.softmax(model(imgs), dim=1)[:, 1].cpu().numpy()
            predictions.extend(probs)
            keys = baselines.keys()
            for i in range(len(probs)):
                b = {k: (baselines[k][i].item() if isinstance(baselines[k][i], torch.Tensor) else baselines[k][i]) for k in keys}
                baselines_list.append(b)
                
    df_eval = pd.DataFrame(baselines_list)
    df_eval['CNN_Prob'] = predictions
    return df_eval


## 7. Full 1:1 Experimentation Loop


In [ ]:
IMAGE_SIZES = [5, 20, 60]
HORIZONS = [5, 20, 60]
ENSEMBLE_SIZE = 5

if TESTING:
    IMAGE_SIZES = [5]
    HORIZONS = [5]
    ENSEMBLE_SIZE = 1
    
results = []

for n_days in IMAGE_SIZES:
    for horizon in HORIZONS:
        print(f"\n{'='*50}\nRunning Configuration: I{n_days} / R{horizon}\n{'='*50}")
        
        train_loader, val_loader, test_loader = create_dataloaders(df_market, n_days, horizon, batch_size=512)
        if train_loader is None or len(train_loader) == 0:
            print("Not enough data to train this configuration. Skipping.")
            continue
            
        all_test_preds = []
        for i in range(ENSEMBLE_SIZE):
            print(f"--- Training Ensemble Model {i+1}/{ENSEMBLE_SIZE} ---")
            model = TrendCNN(n_days=n_days)
            model = train_model(model, train_loader, val_loader, epochs=100, lr=1e-5, patience=2)
            df_eval = generate_predictions(model, test_loader)
            all_test_preds.append(df_eval['CNN_Prob'].values)
            
        avg_preds = np.mean(all_test_preds, axis=0)
        df_eval['CNN_Prob'] = avg_preds
        
        def calc_sharpe(signal_col, df):
            try:
                df['Decile'] = df.groupby('Date')[signal_col].transform(lambda x: pd.qcut(x.rank(method='first'), 10, labels=False))
                high_rets = df[df['Decile'] == 9].groupby('Date')['Fwd_Ret'].mean()
                low_rets = df[df['Decile'] == 0].groupby('Date')['Fwd_Ret'].mean()
                spread_rets = high_rets - low_rets
                
                avg_spread = spread_rets.mean()
                std_spread = spread_rets.std() / np.sqrt(horizon)
                annual_vol = std_spread * np.sqrt(252)
                annual_spread = avg_spread * (252 / horizon)
                
                sharpe = annual_spread / annual_vol if annual_vol != 0 else 0
                return annual_spread, sharpe
            except Exception as e:
                return 0, 0
                
        strategies = ['CNN_Prob', 'MOM', 'STR', 'WSTR', 'TREND']
        for strat in strategies:
            ret, sharpe = calc_sharpe(strat, df_eval.copy())
            results.append({
                'Config': f"I{n_days}/R{horizon}",
                'Strategy': strat,
                'Ann_Return': ret,
                'Ann_Sharpe': sharpe
            })
            
df_results = pd.DataFrame(results)
display(df_results)
df_results.to_csv("final_experiment_results.csv", index=False)
print("\nExperimentation Complete! Results saved to final_experiment_results.csv")



Running Configuration: I5 / R5


Indexing Data (N=5, H=5):   0%|          | 0/375 [00:00<?, ?it/s]

Indexing Data (N=5, H=5):   0%|          | 0/495 [00:00<?, ?it/s]

--- Training Ensemble Model 1/5 ---
